# Zero shot notebook

IDEAS:

- Coger 2 modelos, por ejemplo, Mistral 7B instruc y Qwen del otro notebook

- Probar zero-shot directamente

- Hacerles fine-tuning

- Probar de nuevo zeroshot y ver si hay mejora

La idea en este notebook es probar con una estrategia de zero shot, posiblemente en distintos modelos.

También se puede probar son un prompt más elaborado y otro más sencillito.

In [ ]:
import json
import random
import torch
from utils import load_unsloth_model

## Modelo [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)

Voy a probar si funciona todo con este modelo nuevo de febrero de 2026.

In [ ]:
qwen_3_5_GGUF_path = "unsloth/Qwen3.5-9B-GGUF"
model, tokenizer = load_model_unsloth(qwen_3_5_GGUF_path)

model, tokenizer = load_unsloth_model(qwen_3_5_GGUF_path)

# 2. Cargar los datos
with open('multiple_choice.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def test_model_prediction(level="B2"):
    # Filtrar exámenes por el nivel deseado
    exams_in_level = [e for e in data['exams'] if e['level'] == level]
    if not exams_in_level:
        return "Nivel no encontrado"
    
    # Seleccionar un examen, ejercicio y pregunta al azar
    exam = random.choice(exams_in_level)
    exercise = random.choice(exam['exercises'])['exercise']
    question = random.choice(exercise['questions'])
    
    # Formatear las opciones
    opciones_texto = "\n".join([f"{opt['optionId']}) {opt['text']}" for opt in question['options']])
    
    # Crear el Prompt (Instrucción + Contexto + Pregunta)
    prompt = f"""Eres un experto en lengua española. Basándote ÚNICAMENTE en el texto proporcionado, responde a la pregunta seleccionando la opción correcta.

Texto:
{exercise['text']}

Pregunta: {question['text']}
Opciones:
{opciones_texto}

Instrucción: Indica la letra de la opción correcta y justifica brevemente tu respuesta basándote en el texto.

Respuesta:"""

    # Tokenizar e Inferencia
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Generar (limitamos a 150 tokens para una respuesta directa)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decodificar solo la parte generada (después del prompt)
    full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    respuesta_modelo = full_text.split("Respuesta:")[1].strip()
    
    print(f"=== PRUEBA DE PREDICCIÓN (Nivel {level}) ===")
    print(f"ID Pregunta: {question['questionId']}")
    print(f"\nModelo dice:\n{respuesta_modelo}")

# Ejecutar la prueba
test_model_prediction("B2")